In [1]:
import os
import argparse
import pandas as pd
import re
from collections import defaultdict

In [2]:
class DSU:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0]*n
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return
        if self.rank[rx] < self.rank[ry]:
            self.parent[rx] = ry
        elif self.rank[rx] > self.rank[ry]:
            self.parent[ry] = rx
        else:
            self.parent[ry] = rx
            self.rank[rx] += 1
            
def shorten_name(filename):
    """
    Return the substring between the 4th and 6th underscore (i.e. parts[4:6] joined by '_').
    Example:
      self_9-GCA_048174275.1_A.woodhouseii_AW_366498_pri_1.0_genomic.tsv
      -> parts = [..., 'A.woodhouseii', 'AW', '366498', ...] -> returns 'AW_366498'
    If the filename doesn't have enough underscores, return the filename without extension.
    """
    base = os.path.basename(filename)
    noext = re.sub(r"\.tsv$", "", base, flags=re.IGNORECASE)
    parts = noext.split("_")
    if len(parts) >= 6:
        # return the substring between 4th and 6th underscore -> parts[4] and parts[5]
        return f"{parts[4]}_{parts[5]}"
    else:
        return noext


def find_bed_file(tsv_file, bed_dir):
    """Find the matching BED file for a given self*.tsv file."""
    base = os.path.basename(tsv_file)
    # Remove "self_*-" prefix
    no_prefix = re.sub(r"^self[^-]*-", "", base)
    bed_name = re.sub(r"\.tsv$", ".bed", no_prefix, flags=re.IGNORECASE)
    return os.path.join(bed_dir, bed_name)


def get_diagonal_inversions(df, minlen=1000):
    """Return only diagonal inversions (palindromes)."""
    df["inversion"] = df["strand1"] != df["strand2"]
    inv = df[df["inversion"]].copy()
    inv_diag = inv[(inv["start1"] == inv["start2"]) & (inv["end1"] == inv["end2"])]
    return inv_diag[inv_diag["length1"] >= minlen]


In [3]:
def find_sister_groups(bed_df, inv_diag, include_singletons=True):
    """
    Find sister gene groups with union-find.
    Now also keeps singleton groups if include_singletons=True.
    """
    if bed_df is None or bed_df.empty:
        return [], {}

    bed = bed_df.copy().reset_index().rename(columns={"index": "orig_idx"})
    bed["gene_label"] = bed.apply(
        lambda r: f"{int(r['start'])}_{int(r['end'])}_{r['strand']}_{int(r['orig_idx'])}", axis=1
    )

    n = len(bed)
    label_to_idx = {lab: i for i, lab in enumerate(bed["gene_label"])}
    idx_to_label = {i: lab for lab, i in label_to_idx.items()}

    dsu = DSU(n)
    starts = bed["start"].astype(int).values
    ends = bed["end"].astype(int).values
    strands = bed["strand"].astype(str).values

    for _, inv in inv_diag.iterrows():
        inv_start = int(inv["start1"])
        inv_end   = int(inv["end1"])
        if inv_end <= inv_start:
            continue

        inside_mask = (starts >= inv_start) & (ends <= inv_end)
        idxs_inside = [i for i, flag in enumerate(inside_mask) if flag]
        for i in idxs_inside:
            g_start, g_end, g_strand = starts[i], ends[i], strands[i]
            offset_start = g_start - inv_start
            offset_end   = g_end   - inv_start
            sister_start = inv_end - offset_end
            sister_end   = inv_end - offset_start

            cand_mask = (starts <= sister_end) & (ends >= sister_start) & (strands != g_strand)
            cand_idxs = [j for j, flag in enumerate(cand_mask) if flag]
            for j in cand_idxs:
                dsu.union(i, j)

    groups_map = defaultdict(list)
    for i in range(n):
        root = dsu.find(i)
        groups_map[root].append(i)

    groups = []
    group_members = {}
    for root, members in groups_map.items():
        if not include_singletons and len(members) == 1:
            continue
        labels = [idx_to_label[m] for m in sorted(members)]
        groups.append(sorted(labels))
        group_members[root] = [
            {
                "label": idx_to_label[m],
                "orig_idx": int(bed.loc[m, "orig_idx"]),
                "start": int(bed.loc[m, "start"]),
                "end": int(bed.loc[m, "end"]),
                "strand": bed.loc[m, "strand"]
            }
            for m in sorted(members)
        ]

    return groups, group_members

In [5]:
fpath = "/local/storage/kav67/jays/patchworkplot/IGH/output_AW/pairwise_alignments/self_0-GCA_048174545.1_A.woodhouseii_AW_365326_pri_1.0_genomic.tsv"
df = pd.read_csv(fpath, sep="\t")
df.columns = [c.replace("#", "").replace("%", "").replace("+", "") for c in df.columns]

In [6]:
df

,name1,strand1,start1,end1,length1,name2,strand2,start2,end2,length2,id
0,JBKSHY010000175.1_1_IGH,+,1,82155,82155,JBKSHY010000175.1_1_IGH,+,1,82155,82155,100.0%
1,JBKSHY010000175.1_1_IGH,+,6,3996,3991,JBKSHY010000175.1_1_IGH,+,28,4027,4000,70.8%
2,JBKSHY010000175.1_1_IGH,+,28,4027,4000,JBKSHY010000175.1_1_IGH,+,6,3996,3991,71.3%
3,JBKSHY010000175.1_1_IGH,+,181,5649,5469,JBKSHY010000175.1_1_IGH,+,866,6155,5290,70.2%
4,JBKSHY010000175.1_1_IGH,+,183,4660,4478,JBKSHY010000175.1_1_IGH,+,1016,4927,3912,66.3%
...,...,...,...,...,...,...,...,...,...,...,...
1215,JBKSHY010000175.1_1_IGH,+,71715,72630,916,JBKSHY010000175.1_1_IGH,-,28775,29691,917,74.1%
1216,JBKSHY010000175.1_1_IGH,+,71822,72503,682,JBKSHY010000175.1_1_IGH,-,18294,19028,735,77.4%
1217,JBKSHY010000175.1_1_IGH,+,71908,72583,676,JBKSHY010000175.1_1_IGH,-,44288,45000,713,86.4%
1218,JBKSHY010000175.1_1_IGH,+,71915,72503,589,JBKSHY010000175.1_1_IGH,-,16257,16786,530,86.4%


In [7]:
bed_path = find_bed_file(fpath, "/local/storage/kav67/jays/patchworkplot/IGH/bed_files/")
if bed_path and os.path.exists(bed_path):
    bed_df = pd.read_csv(
        bed_path, sep="\t", header=None, usecols=[1, 2, 5],
        names=["start", "end", "strand"]
    )
else:
    bed_df = pd.DataFrame(columns=["start", "end", "strand"])

In [8]:
bed_df

,start,end,strand
0,10000,10306,+
1,10948,11257,-
2,11451,11733,-
3,11538,11868,+
4,12482,12791,-
...,...,...,...
121,70658,70970,-
122,71146,71443,-
123,71248,71560,+
124,72047,72332,+


In [9]:
sample = shorten_name("self_0-GCA_048174545.1_A.woodhouseii_AW_365326_pri_1.0_genomic.tsv")
inv_diag = get_diagonal_inversions(df, minlen=1000)

In [10]:
sample

'AW_365326'

In [11]:
inv_diag

,name1,strand1,start1,end1,length1,name2,strand2,start2,end2,length2,id,inversion
656,JBKSHY010000175.1_1_IGH,+,9714,11540,1827,JBKSHY010000175.1_1_IGH,-,9714,11540,1827,79.9%,True
668,JBKSHY010000175.1_1_IGH,+,9714,16299,6586,JBKSHY010000175.1_1_IGH,-,9714,16299,6586,85.0%,True
672,JBKSHY010000175.1_1_IGH,+,9861,12930,3070,JBKSHY010000175.1_1_IGH,-,9861,12930,3070,90.4%,True
680,JBKSHY010000175.1_1_IGH,+,9960,14603,4644,JBKSHY010000175.1_1_IGH,-,9960,14603,4644,78.7%,True
701,JBKSHY010000175.1_1_IGH,+,10671,16862,6192,JBKSHY010000175.1_1_IGH,-,10671,16862,6192,78.3%,True
...,...,...,...,...,...,...,...,...,...,...,...,...
1183,JBKSHY010000175.1_1_IGH,+,65798,72630,6833,JBKSHY010000175.1_1_IGH,-,65798,72630,6833,78.9%,True
1190,JBKSHY010000175.1_1_IGH,+,67220,72567,5348,JBKSHY010000175.1_1_IGH,-,67220,72567,5348,80.2%,True
1199,JBKSHY010000175.1_1_IGH,+,68376,72568,4193,JBKSHY010000175.1_1_IGH,-,68376,72568,4193,82.6%,True
1205,JBKSHY010000175.1_1_IGH,+,69518,72678,3161,JBKSHY010000175.1_1_IGH,-,69518,72678,3161,74.2%,True


In [14]:
groups, group_members = find_sister_groups(bed_df, inv_diag)

In [16]:
group_members

{0: [{'label': '10000_10306_+_0',
   'orig_idx': 0,
   'start': 10000,
   'end': 10306,
   'strand': '+'},
  {'label': '10948_11257_-_1',
   'orig_idx': 1,
   'start': 10948,
   'end': 11257,
   'strand': '-'},
  {'label': '11451_11733_-_2',
   'orig_idx': 2,
   'start': 11451,
   'end': 11733,
   'strand': '-'},
  {'label': '11538_11868_+_3',
   'orig_idx': 3,
   'start': 11538,
   'end': 11868,
   'strand': '+'},
  {'label': '12482_12791_-_4',
   'orig_idx': 4,
   'start': 12482,
   'end': 12791,
   'strand': '-'},
  {'label': '13332_13638_+_5',
   'orig_idx': 5,
   'start': 13332,
   'end': 13638,
   'strand': '+'},
  {'label': '14249_14564_-_6',
   'orig_idx': 6,
   'start': 14249,
   'end': 14564,
   'strand': '-'},
  {'label': '14369_14658_+_7',
   'orig_idx': 7,
   'start': 14369,
   'end': 14658,
   'strand': '+'},
  {'label': '14731_15039_-_8',
   'orig_idx': 8,
   'start': 14731,
   'end': 15039,
   'strand': '-'},
  {'label': '14841_15153_+_9',
   'orig_idx': 9,
   'start': 

In [17]:
out_rows = []
for g in groups:
    out_rows.append({
        "sample": sample,
        "sister_group": ";".join(g),
        "size": len(g)
    })
#pd.DataFrame(all_groups).to_csv(args.out, sep="\t", index=False)


In [18]:
out_rows

[{'sample': 'AW_365326',
  'sister_group': '10000_10306_+_0;10948_11257_-_1;11451_11733_-_2;11538_11868_+_3;12482_12791_-_4;13332_13638_+_5;14249_14564_-_6;14369_14658_+_7;14731_15039_-_8;14841_15153_+_9;15719_16025_-_10;15830_16135_+_11;16209_16492_-_12;16297_16606_+_13;17780_18090_-_14;17894_18183_+_15;18334_18635_+_16;19204_19510_+_17;20105_20414_-_18;20613_20896_-_19;20702_21011_+_20;21538_21838_+_21;21670_21932_-_22;22156_22450_-_23;22258_22564_+_24;23133_23440_-_25;23727_24036_+_26;24652_24914_-_27;24757_25030_+_28;25997_26255_-_29;26186_26514_+_30;27054_27360_-_31;27168_27462_+_32;27678_27990_+_33;28387_28699_-_34;28937_29246_+_35;30611_30914_-_36;31298_31604_+_37;32222_32531_-_38;32718_33001_-_39;32806_33113_+_40;33466_33780_-_41;33585_33875_+_42;34016_34299_-_43;34104_34413_+_44;34940_35225_+_45;35048_35357_-_46;35590_35873_-_47;35681_35990_+_48;36641_36950_-_49;37136_37425_-_50;37230_37545_+_51;38159_38471_-_52;38662_38944_-_53;38749_39061_+_54;39675_39987_-_55;39795_40078_+_